# 고객 이탈 분석 & 마케팅 성과 분석

---

## 프로젝트 개요

### 배경
최근 구독 기반 서비스 시장이 급성장하면서 **고객 이탈(Churn)**이 기업의 핵심 과제로 떠오르고 있습니다.  
신규 고객 획득 비용은 기존 고객 유지 비용의 **5~7배**에 달하기 때문에, 이탈을 사전에 예측하고 방지하는 것이 수익성 확보의 핵심입니다.

본 프로젝트에서는 가상의 구독 서비스 회사의 고객 데이터를 분석하여 **이탈 패턴**을 파악하고,  
**마케팅 캠페인의 효과**를 정량적으로 평가합니다.

### 분석 목표
1. **고객 현황 파악**: 전체 고객 구성과 이탈률 현황 분석
2. **이탈 원인 분석**: 이탈 고객의 특성과 패턴을 다각도로 분석
3. **코호트 분석**: 가입 시기별 고객 잔존율 추적
4. **마케팅 성과 평가**: 캠페인별 ROI 및 채널 효율성 분석
5. **전략 제안**: 데이터 기반의 이탈 방지 및 마케팅 최적화 전략 도출

### 데이터 설명
| 파일명 | 설명 | 비고 |
|--------|------|------|
| `customers.csv` | 고객 기본 정보 (800명) | 성별, 연령, 지역, 요금제, 이탈여부 등 |
| `usage_2023.csv` | 2023년 월별 이용 내역 | 이용횟수, 이용시간 |
| `usage_2024.csv` | 2024년 1~6월 이용 내역 | 동일 구조 |
| `campaigns.csv` | 마케팅 캠페인 정보 (10건) | 채널, 예산, 대상고객수 |
| `campaign_responses.csv` | 캠페인 반응 데이터 (~3,000건) | 오픈/클릭/전환 여부 |


---
## 1. 환경 설정 & 데이터 로딩


### TODO: 라이브러리 import & 설정

- `pandas as pd`, `numpy as np`, `matplotlib.pyplot as plt`, `seaborn as sns`
- 한글 폰트 설정 (Windows: Malgun Gothic, Mac: AppleGothic)
- `plt.rc("axes", unicode_minus=False)`
- `DATA_DIR = "../data/"`


In [ ]:
# 아래에 코드를 작성하세요


### TODO: 5개 CSV 파일 읽기

| 변수명 | 파일 | 설명 |
|--------|------|------|
| `customers` | `customers.csv` | 고객 마스터 (800명) |
| `usage_2023` | `usage_2023.csv` | 2023년 월별 이용 내역 |
| `usage_2024` | `usage_2024.csv` | 2024년 월별 이용 내역 |
| `campaigns` | `campaigns.csv` | 마케팅 캠페인 정보 (10개) |
| `responses` | `campaign_responses.csv` | 캠페인 반응 데이터 |

각 파일의 `shape`과 `head(3)`을 출력하세요.


In [ ]:
# 아래에 코드를 작성하세요


### TODO: 데이터 구조 확인

5개 DataFrame에 `info()`를 실행하여 타입과 결측치를 확인하세요.


In [ ]:
# 아래에 코드를 작성하세요


---
## 2. 데이터 품질 확인

### TODO: 품질 문제 탐색

1. **customers 성별**: `customers["성별"].value_counts(dropna=False)` → 남/여 외 이상값 확인
2. **customers 연령**: `customers["연령"].describe()` → min/max 확인
3. **customers 지역**: `customers["지역"].value_counts()` → 공백/오타 확인
4. **customers 월평균결제액**: `dtype` 확인 + 샘플 20개 출력 → 콤마 포함 여부
5. **usage 결측**: `usage_2023.isna().sum()`, `usage_2024.isna().sum()`
6. **responses 중복**: `responses.duplicated().sum()`


In [ ]:
# 아래에 코드를 작성하세요


### 데이터 품질 문제 정리

| 파일 | 문제 유형 | 상세 내용 | 처리 방안 |
|------|-----------|-----------|----------|
| customers | 성별 불일치 | M/F/남/여/남자/여자 등 혼재 | 남/여/미상으로 통일 |
| customers | 연령 이상치 | 음수, 150세 이상 등 비현실적 값 | NaN 처리 후 중앙값 대체 |
| customers | 지역 공백/오타 | 앞뒤 공백, 오타 존재 | strip() 후 통일 |
| customers | 가입일 형식 혼재 | YYYY-MM-DD, YYYY/MM/DD 등 | datetime 통일 변환 |
| customers | 월평균결제액 | 콤마 포함 문자열, NaN | 콤마 제거 후 숫자 변환 |
| campaign_responses | 중복 행 | 동일 레코드 중복 | 중복 제거 |
| campaign_responses | 결측치 | 전환금액 등 일부 결측 | 0으로 대체 |


---
## 3. 데이터 전처리

### TODO: customers 전처리

1. **성별 통일**:
   - `str.strip()` → `replace({"남자":"남", "여자":"여", "M":"남", "F":"여"})` → NaN은 `fillna("미상")`

2. **연령 이상치**:
   - 0 미만 또는 100 초과 → `np.nan`
   - NaN → `fillna(customers["연령"].median())`

3. **지역 통일**:
   - `str.strip()` → `replace({"부산광역시":"부산"})` 등 (먼저 `value_counts()`로 확인)

4. **가입일 변환**: `pd.to_datetime(customers["가입일"], format="mixed")`

5. **월평균결제액**:
   1. `replace()`로 값 치환
2. `pd.to_numeric(errors='coerce')`로 숫자 변환
3. `fillna()`로 결측치 처리


In [ ]:
# 아래에 코드를 작성하세요


### TODO: usage 데이터 전처리

1. 2023 + 2024 합치기: `usage = pd.concat([usage_2023, usage_2024], ignore_index=True)`
2. 결측 처리: `이용횟수`와 `이용시간(분)` NaN → 0으로 `fillna(0)`
3. 결과 확인: `usage.shape`, `usage.isna().sum()`


In [ ]:
# 아래에 코드를 작성하세요


### TODO: campaign_responses 전처리

1. 중복 제거: `responses = responses.drop_duplicates()`
2. 전환금액 NaN → 0: `responses["전환금액"] = responses["전환금액"].fillna(0)`
3. 결과 확인: `responses.shape`


In [ ]:
# 아래에 코드를 작성하세요


---
## 4. 데이터 통합

### TODO: 고객별 이용 내역 집계

고객별로 전체 기간의 이용 행동을 요약합니다.

`고객ID` 기준 groupby + agg:

| 변수명 | 원본 컬럼 | 집계 방법 |
|--------|----------|----------|
| 총이용횟수 | 이용횟수 | sum |
| 평균이용횟수 | 이용횟수 | mean |
| 총이용시간 | 이용시간(분) | sum |
| 이용월수 | 연월 | count |

결과를 `usage_summary`에 저장, `reset_index()` 적용

결과 확인: `usage_summary.shape`, `usage_summary.head()`


In [ ]:
# 아래에 코드를 작성하세요


### TODO: 캠페인 반응 집계 & 최종 통합

1. **responses + campaigns merge**: `pd.merge(responses, campaigns, on="캠페인ID", how="left")`
2. **고객별 캠페인 반응 집계**:
   `고객ID` 기준 groupby + agg:

| 변수명 | 원본 컬럼 | 집계 방법 |
|--------|----------|----------|
| 캠페인수신수 | 캠페인ID | count |
| 총전환금액 | 전환금액 | sum |

결과를 `camp_summary`에 저장, `reset_index()` 적용
3. **최종 통합**: customers + usage_summary + camp_summary를 `고객ID` 기준으로 merge (how="left")
4. 결과 확인: `df.shape`, `df.head()`


In [ ]:
# 아래에 코드를 작성하세요


---
## 5. 고객 현황 분석

### TODO: 전체 현황 요약

아래 지표를 출력하세요:
- 전체 고객 수
- 이탈 고객 수와 이탈률 (이탈여부 == 1)
- 요금제별 고객 수: `customers["요금제"].value_counts()`
- 지역별 고객 수: `customers["지역"].value_counts()`


In [ ]:
# 아래에 코드를 작성하세요


### TODO: 고객 현황 시각화

2×2 서브플롯(`fig, axes = plt.subplots(2, 2, figsize=(12, 8))`):
1. 좌상: 이탈 비율 파이차트 → `이탈여부` value_counts
2. 우상: 요금제별 고객 수 막대그래프
3. 좌하: 지역별 고객 수 막대그래프
4. 우하: 연령 히스토그램


In [ ]:
# 아래에 코드를 작성하세요


---
## 6. 이탈 분석 (핵심)

### TODO: 이탈 vs 유지 고객 비교

이탈여부(0/1)별로 평균값을 비교하세요:
사용할 메서드: `groupby()`, `mean()`, `round()`

어떤 변수에서 이탈/유지 고객의 차이가 큰지 확인하세요.


In [ ]:
# 아래에 코드를 작성하세요


### TODO: 이탈률 비교 대시보드

2×3 서브플롯(`fig, axes = plt.subplots(2, 3, figsize=(18, 10))`):

각 변수별 이탈률을 막대그래프로:
1. 연령대별 이탈률: `df.groupby("연령대")["이탈여부"].mean() * 100`
   - 연령대가 없으면 먼저 생성: `pd.cut(df["연령"], bins=[0,25,35,45,55,100], labels=[...])`
2. 성별 이탈률
3. 요금제별 이탈률
4. 지역별 이탈률
5. 가입경로별 이탈률
6. 이탈 vs 유지 월평균결제액 박스플롯: `sns.boxplot(x="이탈여부", y="월평균결제액")`


In [ ]:
# 아래에 코드를 작성하세요


### TODO: 근속기간과 이탈

1. 근속기간(월) 계산:
   1. `pd.to_datetime()`으로 날짜 변환 (`이탈일` 컬럼)
2. `fillna()`로 결측치 처리
3. `astype()`으로 타입 변환
2. 이탈 vs 유지 근속월수 히스토그램 비교
3. `sns.histplot(data=df, x="근속월수", hue="이탈여부", bins=20)`


In [ ]:
# 아래에 코드를 작성하세요


---
## 7. 코호트 분석

코호트 분석: **같은 시기에 가입한 고객 그룹**의 잔존율을 추적합니다.

### TODO: 코호트 잔존율 테이블 & 히트맵

1. **코호트 지정**: `customers["코호트"] = customers["가입일"].dt.to_period("Q")` (분기별)
2. **usage에 코호트 merge**: `usage = pd.merge(usage, customers[["고객ID","코호트"]], on="고객ID")`
3. **코호트별 활동 고객 수 집계**:
   1. `pivot_table()`로 매트릭스 생성
4. **잔존율 계산**: 각 행의 첫 번째 값(최초 고객수) 대비 비율
   사용할 메서드: `divide()`
5. **히트맵**: `sns.heatmap(retention, annot=True, fmt=".0f", cmap="YlOrRd_r")`


In [ ]:
# 아래에 코드를 작성하세요


### TODO: 코호트 핵심 지표

잔존율 테이블에서 아래 지표를 계산하세요:
- 1개월 후 평균 잔존율
- 3개월 후 평균 잔존율
- 6개월 후 평균 잔존율
- 이탈이 가장 많이 발생하는 시점은?


In [ ]:
# 아래에 코드를 작성하세요


---
## 8. 마케팅 캠페인 성과 분석

### TODO: 캠페인별 KPI 계산

responses와 campaigns를 merge한 뒤, 캠페인별 성과를 집계하세요:

`"캠페인ID", "캠페인명", "채널"` 기준 groupby + agg:

| 변수명 | 원본 컬럼 | 집계 방법 |
|--------|----------|----------|
| 발송수 | 고객ID | count |
| 전환금액 | 전환금액 | sum |

결과를 `campaign_kpi`에 저장, `reset_index()` 적용

campaigns의 예산 정보를 merge해서 **ROI** 계산:
`ROI = 전환금액 / (예산 * 10000) * 100`


In [ ]:
# 아래에 코드를 작성하세요


### TODO: 캠페인 성과 시각화

1행 2열 서브플롯:
1. 좌: 캠페인별 전환율(%) 수평 막대그래프
2. 우: 채널별 평균 ROI 막대그래프


In [ ]:
# 아래에 코드를 작성하세요


---
## 9. 종합 인사이트 & 전략 제안

### TODO: 핵심 지표 요약

print문으로 아래 지표를 출력하세요:
- 전체 고객 수, 이탈 고객 수, 이탈률
- 평균 근속월수 (이탈 vs 유지)
- 이탈률이 가장 높은 요금제/지역/연령대
- 가장 ROI 높은 캠페인/채널


In [ ]:
# 아래에 코드를 작성하세요


### TODO: 최종 대시보드

2×3 서브플롯(`fig, axes = plt.subplots(2, 3, figsize=(18, 10))`):

1. 좌상: 이탈 비율 파이차트
2. 중상: 요금제별 이탈률 막대
3. 우상: 근속월수 이탈/유지 히스토그램
4. 좌하: 코호트 잔존율 히트맵 (sns.heatmap)
5. 중하: 채널별 ROI 막대
6. 우하: 연령대별 이탈률 막대

`fig.suptitle("고객 이탈 & 마케팅 분석 대시보드")`


In [ ]:
# 아래에 코드를 작성하세요


---
## 이탈 방지 전략 제안

데이터 분석 결과를 바탕으로 다음과 같은 전략을 제안합니다.

### 1. 조기 경보 시스템 구축
- **이용 감소 모니터링**: 이탈 고객은 이탈 전 수개월간 이용량이 점진적으로 감소합니다.
  - 전월 대비 이용횟수 30% 이상 감소 시 자동 알림 발생
  - 2개월 연속 감소 시 리텐션 캠페인 자동 발송

### 2. 온보딩 강화 (가입 후 1~3개월)
- 코호트 분석 결과, **가입 초기 1~3개월**이 이탈 집중 구간입니다.
  - 가입 후 1주: 환영 메시지 + 핵심 기능 가이드
  - 가입 후 1개월: 이용 현황 리포트 + 맞춤 추천
  - 가입 후 3개월: 장기 이용 혜택 안내

### 3. 이탈 위험 요금제 고객 관리
- 이탈률이 높은 요금제 고객을 대상으로:
  - 상위 요금제 업그레이드 프로모션
  - 요금제별 차별화된 혜택 강화
  - 가격 대비 가치(Value for Money) 인식 개선

### 4. 가입경로별 차별화 전략
- 이탈률이 높은 가입경로로 유입된 고객에게:
  - 추가 온보딩 프로세스 적용
  - 가입 동기에 맞는 맞춤 콘텐츠 제공

---

## 마케팅 효율화 제안

### 1. 고성과 채널 집중 투자
- ROI가 높은 채널에 예산을 재배분합니다.
- 저성과 채널은 크리에이티브/타겟팅 개선 후 재테스트합니다.

### 2. 이탈 위험 고객 타겟 캠페인
- 이탈 위험 고객은 일반 캠페인에 대한 반응률이 낮으므로:
  - **개인화된 할인 쿠폰** (전환율 향상)
  - **1:1 상담 연결** (고관여 유도)
  - **경쟁사 대비 장점 강조** 메시지

### 3. 캠페인 퍼널 최적화
- 오픈율은 높지만 클릭율이 낮은 캠페인: **CTA(Call-to-Action) 개선**
- 클릭율은 높지만 전환율이 낮은 캠페인: **랜딩페이지 개선**
- A/B 테스트를 통한 지속적인 성과 개선

### 4. 코호트 기반 리마케팅
- 가입 시기별로 잔존율이 다르므로, 코호트별 맞춤 캠페인 설계
- 잔존율이 급락하는 시점에 선제적 리텐션 캠페인 발송

---

### 프로젝트 요약

본 분석을 통해 고객 이탈의 주요 원인과 패턴을 파악하고, 마케팅 캠페인의 효과를 정량적으로 평가했습니다.  
**데이터 기반의 의사결정**을 통해 이탈률을 낮추고 마케팅 ROI를 개선할 수 있는 구체적인 전략을 도출했습니다.

> "측정할 수 없으면, 관리할 수 없다." - 피터 드러커
